In [6]:
from reader2 import EpubBilingualParser
import pandas as pd 
parser = EpubBilingualParser("./data")
data2 = parser.get_all_books_data()

In [7]:
data2

{'Catriona.epub': {'CHAPTER I — A BEGGAR ON HORSEBACK': [{'en': 'The 25th day of August, 1751, about two in the afternoon, I, David Balfour, came forth of the British Linen Company, a porter attending me with a bag of money, and some of the chief of these merchants bowing me from their doors.',
    'ru': '25 августа 1751 года около двух часов дня, я, Дэвид Бэлфур, вышел из банка Британского Льнопрядильного кредитного общества; рядом шел рассыльный с мешком денег, а важные коммерсанты, стоя в дверях, провожали меня поклонами.'},
   {'en': 'Two days before, and even so late as yestermorning, I was like a beggar-man by the wayside, clad in rags, brought down to my last shillings, my companion a condemned traitor, a price set on my own head for a crime with the news of which the country rang.',
    'ru': 'Всего два дня назад, и даже еще вчера утром, я ничем не отличался от нищего бродяги, ходил в лохмотьях, без единого шиллинга в кармане, товарищем моим был приговоренный к виселице изменни

In [2]:
from reader2 import EpubBilingualParser
import pandas as pd 
parser = EpubBilingualParser("./data")
BOOKS_DIR = './corpus/books' 
data = parser.process_epub_files_bs4_v2(BOOKS_DIR)

Поиск EPUB-файлов в: ./corpus/books

--- Обработка книги: alice_in_wonderland_en_ru ---
  -> Извлечена глава: Неизвестная Глава (chapter1.html) (Англ: 7, Рус: 7)
  -> Извлечена глава: Chapter I. Down the Rabbit-Hole / Глава I. Кроличья нора. (Англ: 63, Рус: 63)
  -> Извлечена глава: Chapter II. The Pool of Tears / Глава II. Пруд из слёз. (Англ: 56, Рус: 56)
  -> Извлечена глава: Chapter III. A Caucus-Race and a Long Tale / Глава III. Марафонский бег и история с концом. (Англ: 67, Рус: 67)
  -> Извлечена глава: Chapter IV. The Rabbit Sends in a Little Bill / Глава IV. Белый Кролик и его дом. (Англ: 91, Рус: 90)
  -> Извлечена глава: Chapter V. Advice from a Caterpillar / Глава V. Гусеница и её совет. (Англ: 88, Рус: 83)
  -> Извлечена глава: Chapter VI. Pig and Pepper / Глава VI. Поросёнок и перец. (Англ: 108, Рус: 108)
  -> Извлечена глава: Chapter VII. A Mad Tea-Party / Глава VII. За чашкой чая. (Англ: 124, Рус: 124)
  -> Извлечена глава: Chapter VIII. The Queen’s Croquet-Ground / Гла

In [ ]:
import os
import re

def get_agent_translation(book_name: str, agent_index: int, chapter_name: str) -> str:
    """
    Находит и читает сохраненный файл перевода для конкретной главы и агента.
    """
    clean_book = re.sub(r'[\\/*?:"<>|]', "", book_name).strip()
    clean_chap = re.sub(r'[\\/*?:"<>|]', "", chapter_name).strip()
    
    # 2. Собираем путь: output / Название_Книги / Номер_Агента / Название_Главы.txt
    file_path = os.path.join("output", clean_book, str(agent_index), f"{clean_chap}.txt")
    
    if not os.path.exists(file_path):
        return None
    
    try:
        with open(file_path, "r", encoding="utf-8") as f:
            content = f.read()
            
            return content
            
    except Exception as e:
        print(f"Ошибка при чтении файла {file_path}: {e}")
        return None
    
def get_agent_translation_v2(book_name: str, agent_index: int, chapter_name: str) -> str:
    """
    Находит и читает файл перевода для структуры:
    agents / agent_index / book_name / chapter_name.txt
    """
    clean_book = re.sub(r'[\\/*?:"<>|]', "", book_name).strip()
    clean_chap = re.sub(r'[\\/*?:"<>|]', "", chapter_name).strip()
    
    # agents -> 0 -> alice_in_wonderland_en_ru -> chapter.txt
    file_path = os.path.join("agents", str(agent_index), clean_book, f"{clean_chap}.txt")

    if not os.path.exists(file_path):
        return None
    
    try:
        with open(file_path, "r", encoding="utf-8") as f:
            content = f.read()
            
            # 3. Обработка разделителей метрик
            if "--- METRICS ---" in content:
                # Используем ваш разделитель из 15 дефисов
                parts = content.split("---------------")
                if len(parts) > 1:
                    content = parts[-1].strip()
            
            return content
            
    except Exception as e:
        print(f"Ошибка при чтении файла {file_path}: {e}")
        return None

In [ ]:
import pandas as pd
import torch
import spacy
from fuzzywuzzy import fuzz
import evaluate
from comet.models import download_model, load_from_checkpoint
from collections import Counter

class TranslationEvaluator:
    def __init__(self, model="ru_core_news_lg"):
        print("--- Инициализация моделей на GPU ---")
        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        print("Загрузка NLP моделей...")
        self.nlp_ru = spacy.load(model)
        
        # Список русских дискурсивных связок (коннекторов)
        # Эти слова отвечают за логику и связность текста
        self.connectives = {
            'однако', 'следовательно', 'поэтому', 'впрочем', 'зато', 
            'напротив', 'кроме того', 'в то время как', 'несмотря на', 
            'хотя', 'значит', 'итак', 'тем не менее', 'поскольку'
        }
        
        print("Загрузка COMET...")
        comet_path = download_model("Unbabel/wmt22-comet-da")
        self.comet_model = load_from_checkpoint(comet_path)
        
        print("Загрузка BLEURT-20...")
        self.bleurt = evaluate.load("bleurt", config_name="BLEURT-20")
    

        print("Загрузка BERTScore...")
        self.bert_score = evaluate.load("bertscore")

    

    def _extract_features(self, text):
        "for blonde"
        doc = self.nlp_ru(text)
        features = {
            "P": [], 
            "T": [], 
            "E": [], 
            "C": []  
        }

        for token in doc:
            # 1. Извлекаем местоимения
            if token.pos_ == "PRON":
                features["P"].append(token.text.lower())
            
            # 2. Извлекаем время глаголов (прошедшее, настоящее, будущее)
            if token.pos_ in ["VERB", "AUX"]:
                tense = token.morph.get("Tense")
                if tense:
                    features["T"].append(tense[0])
            
            # 3. Извлекаем связки (проверка по списку)
            if token.text.lower() in self.connectives:
                features["C"].append(token.text.lower())

        # 4. Извлекаем сущности (имена, организации, локации)
        features["E"] = [ent.text.lower() for ent in doc.ents]
        
        return features

    def calculate_f1(self, cand_list, ref_list):
        "for blonde"
        if not ref_list:
            return 1.0 if not cand_list else 0.0
            
        c_counts = Counter(cand_list)
        r_counts = Counter(ref_list)
        
        matches = sum((c_counts & r_counts).values())
        
        precision = matches / sum(c_counts.values()) if c_counts else 0
        recall = matches / sum(r_counts.values()) if r_counts else 0
        
        if precision + recall > 0:
            return 2 * (precision * recall) / (precision + recall)
        return 0.0

    def score(self, candidate, reference):
        "for blonde"
        cand_feats = self._extract_features(candidate)
        ref_feats = self._extract_features(reference)
        
        results = {}
        for cat in ["P", "T", "E", "C"]:
            results[cat] = self.calculate_f1(cand_feats[cat], ref_feats[cat])
        
        # Итоговый балл — среднее по 4 категориям
        final_score = sum(results.values()) / len(results)
        return final_score, results

    def evaluate_dataframe(self, df, batch_size=16):
        """Основной метод оценки всего датафрейма"""
        sources = df['src'].tolist()
        predictions = df['mt'].tolist()
        references = df['ref'].tolist()
        
        print(f"Начинаю расчет для {len(df)} строк...")

        # 1. Расчет COMET 
        print("Вычисляю COMET...")
        data_comet = [{"src": s, "mt": p, "ref": r} for s, p, r in zip(sources, predictions, references)]
        comet_output = self.comet_model.predict(data_comet, batch_size=batch_size, gpus=1)
        df['COMET'] = comet_output.scores


        # 2. Расчет BLEURT
        print("Вычисляю BLEURT...")
        bleurt_res = self.bleurt.compute(predictions=predictions, references=references)
        df['BLEURT'] = bleurt_res['scores']

        # 3. Расчет BERTScore 
        print("Вычисляю BERTScore...")
        bs_res = self.bert_score.compute(predictions=predictions, references=references, lang="ru", device=self.device)
        df['BERTScore_F1'] = bs_res['f1']

        # 4. Расчет BLonde
        print("Вычисляю BLONDE...")
        blonde_scores = []
        for cand, ref in zip(predictions, references):
            score, _ = self.score(cand, ref)
            blonde_scores.append(score)
            
        df['BLONDE'] = blonde_scores

        return df


c:\Users\andre\Documents\rag_test\.venv\Lib\site-packages\torchmetrics\utilities\imports.py:23: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import DistributionNotFound, get_distribution


In [ ]:
import os
import sys

os.environ["TRANSFORMERS_OFFLINE"] = "0"

# Подмена версий в метаданных до импорта
class MockDist:
    def __init__(self, version): self.version = version
    @property
    def metadata(self): return {"Version": self.version}

import importlib.metadata
original_version = importlib.metadata.version

def patched_version(package):
    try:
        v = original_version(package)
        return v if v is not None else "2.6.0"
    except:
        return "2.6.0"

importlib.metadata.version = patched_version

from transformers import pipeline, AutoModelForTokenClassification, AutoTokenizer

model_name = "Babelscape/wikineural-multilingual-ner"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForTokenClassification.from_pretrained(model_name)

nlp = pipeline("ner", model=model, tokenizer=tokenizer, aggregation_strategy="simple")

c:\Users\andre\Documents\rag_test\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Пытаемся загрузить Data-Lab...


Device set to use cuda:0


In [76]:
evaluator= TranslationEvaluator()

--- Инициализация моделей на GPU ---
Загрузка NLP моделей...
Загрузка COMET...


Fetching 5 files: 100%|██████████| 5/5 [00:00<?, ?it/s]
Lightning automatically upgraded your loaded checkpoint from v1.8.3.post1 to v2.6.0. To apply the upgrade to your files permanently, run `python -m pytorch_lightning.utilities.upgrade_checkpoint C:\Users\andre\.cache\huggingface\hub\models--Unbabel--wmt22-comet-da\snapshots\2760a223ac957f30acfb18c8aa649b01cf1d75f2\checkpoints\model.ckpt`


Загрузка BLEURT-20...


c:\Users\andre\Documents\rag_test\.venv\Lib\site-packages\keras\src\export\tf2onnx_lib.py:8: FutureWarning: In the future `np.object` will be defined as the corresponding NumPy scalar.
  if not hasattr(np, "object"):



INFO:tensorflow:Reading checkpoint C:\Users\andre\.cache\huggingface\metrics\bleurt\BLEURT-20\downloads\extracted\ee7a56b1c3595b5497591a490315729cd1af480a5aa7ae3a85cbe946f4b05d42\BLEURT-20.
INFO:tensorflow:Config file found, reading.
INFO:tensorflow:Will load checkpoint BLEURT-20
INFO:tensorflow:Loads full paths and checks that files exists.
INFO:tensorflow:... name:BLEURT-20
INFO:tensorflow:... bert_config_file:bert_config.json
INFO:tensorflow:... max_seq_length:512
INFO:tensorflow:... vocab_file:None
INFO:tensorflow:... do_lower_case:None
INFO:tensorflow:... sp_model:sent_piece
INFO:tensorflow:... dynamic_seq_length:True
INFO:tensorflow:Creating BLEURT scorer.
INFO:tensorflow:Creating SentencePiece tokenizer.
INFO:tensorflow:Creating SentencePiece tokenizer.
INFO:tensorflow:Will load model: C:\Users\andre\.cache\huggingface\metrics\bleurt\BLEURT-20\downloads\extracted\ee7a56b1c3595b5497591a490315729cd1af480a5aa7ae3a85cbe946f4b05d42\BLEURT-20\sent_piece.model.
INFO:tensorflow:Sentenc

INFO:tensorflow:BLEURT initialized.


Загрузка BERTScore...


In [75]:
!pip install git+https://github.com/google-research/bleurt.git

  Cloning https://github.com/google-research/bleurt.git to c:\users\andre\appdata\local\temp\pip-req-build-vr1tmdci
  Resolved https://github.com/google-research/bleurt.git to commit cebe7e6f996b40910cfaa520a63db47807e3bf5c
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
   ---------------------------------------- 0.0/331.8 MB ? eta -:--:--
   ---------------------------------------- 3.9/331.8 MB 21.5 MB/s eta 0:00:16
   - -------------------------------------- 8.4/331.8 MB 21.7 MB/s eta 0:00:15
   - -------------------------------------- 13.6/331.8 MB 23.8 MB/s eta 0:00:14
   -- ------------------------------------- 21.0/331.8 MB 28.8 MB/s eta 0:00:11
   --- ------------------------------------ 26.2/331.8 MB

  Running command git clone --filter=blob:none --quiet https://github.com/google-research/bleurt.git 'C:\Users\andre\AppData\Local\Temp\pip-req-build-vr1tmdci'
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
unbabel-comet 2.2.7 requires protobuf<5.0.0,>=4.24.4, but you have protobuf 6.33.4 which is incompatible.

[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
from mawo_pymorphy3 import create_analyzer
import nltk
from nltk.stem import WordNetLemmatizer
nltk.download('wordnet', quiet=True)
wnl = WordNetLemmatizer()
analyzer = create_analyzer()
def split_entities(text, lang='ru'):
    doc = nlp(text)
    
    entities = {
        "PER": set(),
    }
    
    for ent in doc:
        if ent['entity_group'] in ["PER",'PERSON']:

            word_text = ent['word'].strip()
            if lang == 'ru':
                word_info = analyzer.parse(word_text)[0]
                norm_name = word_info.normal_form.title()
                entities["PER"].add(norm_name)
            else:
                words = word_text.split()
                lemmatized_words = [wnl.lemmatize(w.lower(), pos='n').title() for w in words]
                norm_name = " ".join(lemmatized_words)
                
            if len(norm_name) > 1:
                entities["PER"].add(norm_name)

            
    return {k: list(v) for k, v in entities.items()}

In [79]:
def evaluate_agents(data_epub, agent_indices, evaluator):
    all_rows = []
    
    for agent_id in agent_indices:
        for book, chapters in data_epub.items():
            i = 0
            for chap_name, pairs in chapters.items():
                if pairs['en'] == "":
                    continue
                mt = get_agent_translation_v2(book, agent_id, str(i))
                i+=1
                # mt = get_agent_translation(book, agent_id, chap_name)
                if not mt: continue
                src = pairs['en']
                ref = pairs['ru']
                # src = "\n".join([p['en'] for p in pairs])
                # ref = "\n".join([p['ru'] for p in pairs])
                
                all_rows.append({
                    "Agent": agent_id,
                    "book": book,
                    "Chapter": i,
                    "src": src,
                    "mt": mt,
                    "ref": ref,
                    'ner_ru_ref': split_entities(ref,'ru'),
                    'ner_ru_mt': split_entities(mt,'ru'),
                    'ner_en':   split_entities(src,'en')
                })

    df = pd.DataFrame(all_rows)

    return  evaluator.evaluate_dataframe(df)

In [80]:
result = evaluate_agents(data,[0,1,2,3,4],evaluator)

💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
You are using a CUDA device ('NVIDIA GeForce RTX 4070') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more details, read https://pytorch.org/docs/stable/generated/torch.set_float32_matmul_precision.html#torch.set_float32_matmul_precision
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Начинаю расчет для 372 строк...
Вычисляю COMET...


Predicting DataLoader 0: 100%|██████████| 24/24 [00:37<00:00,  1.55s/it]


Вычисляю BLEURT...
Вычисляю BERTScore...
Вычисляю BLONDE...


In [83]:
agent_means = result.groupby('Agent')[['BLEURT','COMET','BERTScore_F1']].mean()

In [84]:
agent_means

,BLEURT,COMET,BERTScore_F1
Agent,,,
0,0.536817,0.867627,0.800553
1,0.528495,0.853486,0.790850
2,0.538507,0.866113,0.798915
3,0.532155,0.864528,0.799173
4,0.536501,0.866165,0.799021


In [43]:
result['PER_list_ru_mt'] = result['ner_ru_mt'].apply(lambda x: x.get('PER', []))
result['PER_list_ru_ref'] = result['ner_ru_ref'].apply(lambda x: x.get('PER', []))
result['PER_list_en'] = result['ner_en'].apply(lambda x: x.get('PER', []))

In [ ]:
import pandas as pd
import torch
from sentence_transformers import SentenceTransformer, util

# модель ( EN <-> RU)
model = SentenceTransformer('sentence-transformers/LaBSE')

def get_best_vector_match(en_name, ru_set):
    if not ru_set or pd.isna(en_name) or len(ru_set) == 0:
        return "None"
    candidates = list(ru_set)
    
    en_emb = model.encode(str(en_name), convert_to_tensor=True)
    ru_embs = model.encode(candidates, convert_to_tensor=True)
    
    scores = util.cos_sim(en_emb, ru_embs)[0]
    best_idx = torch.argmax(scores).item()
    
    if scores[best_idx].item() < 0.4:
        return "None"
        
    return candidates[best_idx]

df_survival = result.explode('PER_list_en').copy()
df_survival = df_survival[df_survival['PER_list_en'].notna()]


df_survival['match_mt'] = df_survival.apply(
    lambda r: get_best_vector_match(r['PER_list_en'], r['PER_list_ru_mt']), axis=1
)

df_survival['match_ref'] = df_survival.apply(
    lambda r: get_best_vector_match(r['PER_list_en'], r['PER_list_ru_ref']), axis=1
)

In [58]:
import pandas as pd

def get_canonical_name(series):
    valid_names = series[series != "None"]
    if valid_names.empty:
        return "None"
    return valid_names.mode()[0]

canonical_map = df_survival.groupby(['Agent', 'PER_list_en'])['match_mt'].apply(get_canonical_name).reset_index()
canonical_map.rename(columns={'match_mt': 'canonical_name'}, inplace=True)

df_stability = df_survival.merge(canonical_map, on=['Agent', 'PER_list_en'], how='left')

df_stability['is_stable'] = (df_stability['match_mt'] == df_stability['canonical_name']).astype(int)

In [59]:
mask_not_none = (df_stability['match_mt'] != "None")

agent_rating = df_stability[mask_not_none].groupby('Agent')['is_stable'].agg([
    ('accuracy_stability', 'mean'), 
    ('found_count', 'count')        
]).reset_index()

agent_rating['accuracy_pct'] = (agent_rating['accuracy_stability'] * 100).round(2)

print("--- Рейтинг (только по найденным именам) ---")
print(agent_rating.sort_values(by='accuracy_pct', ascending=False))

--- Рейтинг (только по найденным именам) ---
   Agent  accuracy_stability  found_count  accuracy_pct
4      4            0.940741          135         94.07
3      3            0.933824          136         93.38
1      1            0.918699          123         91.87
2      2            0.912409          137         91.24
0      0            0.902256          133         90.23


In [60]:
one_chapter_book = ['fyrtoiet_en_ru','the_gift_of_the_magi_en_ru','the_raven_en_ru']

mask_clean = (df_stability['match_mt'] != "None") & (~df_stability['book'].isin(one_chapter_book))

agent_rating = df_stability[mask_clean].groupby('Agent')['is_stable'].agg([
    ('accuracy_stability', 'mean'), 
    ('found_count', 'count')        
]).reset_index()

agent_rating['accuracy_pct'] = (agent_rating['accuracy_stability'] * 100).round(2)

print(f"--- Рейтинг (исключены книги: {', '.join(one_chapter_book)}) ---")
print(agent_rating.sort_values(by='accuracy_pct', ascending=False))

--- Рейтинг (исключены книги: fyrtoiet_en_ru, the_gift_of_the_magi_en_ru, the_raven_en_ru) ---
   Agent  accuracy_stability  found_count  accuracy_pct
4      4            0.937500          128         93.75
3      3            0.930769          130         93.08
1      1            0.914530          117         91.45
2      2            0.908397          131         90.84
0      0            0.897638          127         89.76


In [ ]:
one_chapter_book = ['fyrtoiet_en_ru', 'the_gift_of_the_magi_en_ru', 'the_raven_en_ru']

df_survival_filtered = df_survival[~df_survival['book'].isin(one_chapter_book)]

survival_pivot = df_survival_filtered.pivot_table(
    index=['Agent', 'PER_list_en'], 
    columns='Chapter', 
    values='match_mt', 
    aggfunc='first'
).fillna("None")

print(f"--- Таблица выживаемости (без {len(one_chapter_book)} коротких книг) --- ОЧень Много None чтобы мы могли развернуть")
survival_pivot.head(30)

--- Таблица выживаемости (без 3 коротких книг) ---


Chapter                                   1      2          3        4   \
Agent PER_list_en                                                         
0     ##Ager Duchess         Огастес Дампьер   None       None     None   
      ##Iar John                        None   None       None  Иоанн Я   
      ##Iar Laurence                    None   None       None  Лоренцо   
      ##Man                             None   None       None     None   
      ##N Balthasar                Бальтазар   None       None     None   
      ##N Friar John              Брат Иоанн   None       None     None   
      ##Ter                             None   None       None     None   
      ##Ts                              None   None       None     None   
      ##Ule                             None   None       None     None   
      ##Ulet                            None   None       None     None   
      ##Y                               None   None       None        Ф   
      Abraham                Джульетты Абрам   None       None     None   
      Ali                               None   None       None     None   
      Alice                             None  Алисе      Алиса    Алисе   
      Alidoro                           None   None       None     None   
      Alike                             None   None       None     None   
      Augustus Dampier       Огастес Дампьер   None       None     None   
      Balthasar                         None   None  Бальтазар     None   
      Bravo                             None  Браво       None     None   
      Canterville                 Кантервиль   None       None     None   
      Canterville Chase           Кантервиль   None       None     None   
      Cap                               None   None       None     None   
      Capulet                      Капулетть   None       None     None   
      Capulet Gregory    Капулетти Грегореть   None       None     None   
      Capulet Juliet     Капулетти Джульетта   None       None     None   
      Capulet Nurse      Капулетти Джульетта   None       None     None   
      Capulet Peter           Капулётти Петр   None       None     None   
      Capulet Romeo          Капулетти Ромео   None       None     None   
      Cat                               None   None       None     None   
      Cricket                           None   None       None     None   

Chapter                         5      6               7      8      9   \
Agent PER_list_en                                                         
0     ##Ager Duchess          None   None            None   None   None   
      ##Iar John              None   None            None   None   None   
      ##Iar Laurence          None   None            None   None   None   
      ##Man                   None   None            None   None   None   
      ##N Balthasar           None   None            None   None   None   
      ##N Friar John          None   None            None   None   None   
      ##Ter                   None   None            None   None   None   
      ##Ts                    None   None            None   None   None   
      ##Ule                   None   None          ##Мить   None   None   
      ##Ulet                  None   None          ##Мить   None   None   
      ##Y                     None   None            None   None   None   
      Abraham                 None   None            None   None   None   
      Ali                     None   None            None   None   None   
      Alice                  Алиса  Алиса           Алиса  Алиса  Алисе   
      Alidoro                 None   None            None   None   None   
      Alike                   None   None            None     Об   None   
      Augustus Dampier        None   None  Август Дампира   None   None   
      Balthasar          Бальтазар   None            None   None   None   
      Bravo                   None   None            None   None   None   
      Canterville       

In [ ]:
survival_pivot = df_survival_filtered.pivot_table(
    index=['book', 'Agent', 'PER_list_en'], 
    columns='Chapter', 
    values='match_mt', 
    aggfunc='first'
).fillna("None")

In [ ]:
survival_pivot # Хороший пример Август Дампира по агентам

Chapter                                                                                1   \
book                        Agent PER_list_en                                               
alice_in_wonderland_en_ru   0     ##Ter                                              None   
                                  ##Y                                                None   
                                  Alice                                              None   
                                  Dinah                                              None   
                                  Edwin                                              None   
                                  Gryphon                                            None   
                                  Hare                                               None   
                                  Hatter                                             None   
                                  Imperious Prima                                    None   
                                  Knave                                              None   
                                  Liza                                               None   
                                  Lory                                               None   
                                  March Hare                                         None   
                                  Mary Ann                                           None   
                                  Morcar                                             None   
                                  Rabbit                                             None   
                                  Stigand                                            None   
                                  Tertia                                           Терция   
                                  White Rabbit                                       None   
                                  William The Conqueror                              None   
                            1     ##Ter                                              None   
                                  ##Y                                                None   
                                  Alice                                              None   
                                  Dinah                                              None   
                                  Edwin                                              None   
                                  Gryphon                                            None   
                                  Hare                                               None   
                                  Hatter                                             None   
                                  Imperious Prima                                    None   
                                  Knave                                              None   
                                  Liza                                               None   
                                  Lory                                               None   
                                  March Hare                                         None   
                                  Mary Ann                                           None   
                                  Morcar                                             None   
                                  Rabbit                                             None   
                                  Stigand                                            None   
                                  Tertia                                             None   
                                  White Rabbit                                       None   
                                  William The Conqueror                              None   
                            2     ##Ter                                              None   
 

In [16]:
pd.set_option('display.max_rows',575)

In [147]:
result['PER_list_ru_mt'] = result['ner_ru_mt'].apply(lambda x: x.get('PER', []))
result['PER_list_ru_ref'] = result['ner_ru_ref'].apply(lambda x: x.get('PER', []))
result['PER_list_en'] = result['ner_en'].apply(lambda x: x.get('PER', []))